<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/aprepare2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==== Transform outputs from main notebook into app artefacts (NO TRAINING) ====
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import math, re, shutil, glob

# -------- PATHS (adjust if your layout differs) --------
DRIVE_ROOT  = Path("/content/drive/MyDrive/gt-markets")
SRC_ROOT    = DRIVE_ROOT / "app-demo" / "extracted"              # where you moved outputs
DST_ROOT    = DRIVE_ROOT / "AppDemo" / "artefacts_from_mainnb"   # clean app-ready output
DATA_PROCESSED = DRIVE_ROOT / "data" / "processed"               # fallback for Close price

# -------- CONFIG --------
ASSETS = ["GOLD", "BTC", "OIL", "USDCNY"]
FREQ_DIRS = {"daily":"D", "weekly":"W"}  # folder -> freq
ANNUALIZATION = {"D":252, "W":52}
COST_BPS = 5.0

# Map processed data columns for Close fallback if preds don't include Close
PRICE_COL = {
    "GOLD":   "GC=F Close",
    "BTC":    "BTC-USD Close",
    "OIL":    "CL=F Close",
    "USDCNY": "USDCNY=X Close",
}

# Heuristics to detect keywords vs baseline by path/name
def is_keywords_path(p: Path) -> bool:
    s = str(p).lower()
    return any(k in s for k in ["kw", "keyword", "trends", "googletrends", "gt_"])

def infer_asset(name: str) -> str|None:
    n = name.lower()
    # explicit
    for a in ASSETS:
        if a.lower() in n: return a
    # tickers -> asset
    for tick, a in {"gc=f":"GOLD","btc-usd":"BTC","cl=f":"OIL","usdcny=x":"USDCNY"}.items():
        if tick in n: return a
    return None

def infer_strategy(name: str) -> str:
    n = name.lower()
    # common model/strategy tokens
    for s in ["rf","xgb","xgboost","mlp","lstm","gru","lr","svm","ridge","stack","ensemble","calibrated","randforest","randomforest"]:
        if s in n: return s.upper() if len(s)<=4 else s
    if "trend" in n: return "S1_trend"
    if "breakout" in n or "donch" in n: return "S3_breakout"
    if "mean" in n and "rev" in n: return "S2_meanrev"
    return "MODEL"

def latest_processed_csv() -> Path|None:
    files = sorted(DATA_PROCESSED.glob("merged_financial_trends_data_*.csv"))
    return files[-1] if files else None

def to_equity(close: pd.Series, pos: pd.Series, freq: str):
    close = pd.to_numeric(close, errors="coerce")
    ret = close.pct_change().fillna(0.0)
    pos = pos.fillna(0.0).clip(0,1)
    trades = pos.diff().abs().fillna(0.0)
    cost = trades * (COST_BPS/1e4)
    strat = (pos.shift(1) * ret) - cost
    eq = (1 + strat).cumprod()
    ann = ANNUALIZATION[freq]
    mu = strat.mean() * ann
    sd = strat.std() * (ann**0.5) if strat.std()>0 else np.nan
    sharpe = mu / sd if (sd and sd>0) else np.nan
    maxdd = (eq / eq.cummax() - 1).min()
    return eq, {"Return_Ann": mu, "Vol_Ann": sd, "Sharpe": sharpe, "MaxDD": maxdd}

def plot_eq(eq: pd.Series, title: str, outpath: Path):
    fig, ax = plt.subplots(figsize=(6,3))
    eq.plot(ax=ax); ax.set_title(title); ax.grid(True, alpha=0.3)
    fig.savefig(outpath, dpi=150, bbox_inches="tight"); plt.close(fig)

# Prepare destination
if DST_ROOT.exists(): shutil.rmtree(DST_ROOT)
DST_ROOT.mkdir(parents=True, exist_ok=True)

# Fallback prices if needed
price_df = None
pp = latest_processed_csv()
if pp:
    try:
        price_df = pd.read_csv(pp, parse_dates=["Date"]).set_index("Date").sort_index()
    except Exception:
        price_df = None

# Walk daily/weekly
for freq_dir, F in FREQ_DIRS.items():
    src = SRC_ROOT / freq_dir
    if not src.exists():
        print(f"[WARN] Missing {src}")
        continue

    # Candidates: CSVs that look like predictions or signals
    csvs = list(src.rglob("*.csv"))
    pngs = list(src.rglob("*.png")) + list(src.rglob("*.jpg"))

    # Optional overall summary if present (used to augment metrics)
    summary = next((c for c in csvs if any(k in c.name.lower() for k in ["backtest_summary","leaderboard"])), None)
    summary_df = None
    if summary:
        try:
            summary_df = pd.read_csv(summary)
        except Exception:
            summary_df = None

    # Per-asset metrics aggregators
    metrics_rows_baseline = {a: [] for a in ASSETS}
    metrics_rows_keywords = {a: [] for a in ASSETS}

    for c in csvs:
        name = c.name.lower()
        # Skip summary here; handled separately
        if summary and c == summary:
            continue

        # Already-normalized signals? copy with minimal rename
        if name.startswith("signals_") and name.endswith(".csv"):
            asset = infer_asset(name) or infer_asset(str(c))
            if not asset: continue
            strat = infer_strategy(name)
            kind = "keywords" if is_keywords_path(c) else "baseline"
            out_dir = DST_ROOT / asset; (out_dir/"figs").mkdir(parents=True, exist_ok=True)
            dst = out_dir / f"signals_{strat}_{F}.csv"
            shutil.copy2(c, dst)
            # No metrics from signals alone; we’ll infer later if possible
            continue

        # Prediction-like CSV: must have Date + (prob_up or position), and ideally Close
        if any(k in name for k in ["pred","prob","proba","prediction","inference"]):
            asset = infer_asset(name) or infer_asset(str(c))
            if not asset: continue
            kind = "keywords" if is_keywords_path(c) else "baseline"
            strat = infer_strategy(name)

            try:
                dfp = pd.read_csv(c)
            except Exception:
                continue

            # Date column
            dt = next((x for x in ["Date","date","timestamp","time"] if x in dfp.columns), None)
            if dt is None:
                continue
            dfp[dt] = pd.to_datetime(dfp[dt], errors="coerce")
            dfp = dfp.dropna(subset=[dt]).sort_values(dt)

            # Position
            pos_col = next((x for x in ["position","Position"] if x in dfp.columns), None)
            if pos_col is not None:
                pos = pd.to_numeric(dfp[pos_col], errors="coerce").fillna(0.0).clip(0,1)
            else:
                pcol = next((x for x in ["prob_up","p_up","proba","prob","prob1","p1"] if x in dfp.columns), None)
                if pcol is None:
                    continue
                pos = (pd.to_numeric(dfp[pcol], errors="coerce") > 0.5).astype(float)

            # Close
            ccol = next((x for x in ["Close","close","price","close_adj","adj_close"] if x in dfp.columns), None)
            close = None
            if ccol:
                close = pd.to_numeric(dfp[ccol], errors="coerce")
            elif price_df is not None:
                # fallback from processed data
                col = PRICE_COL.get(asset)
                if col in price_df.columns:
                    close = price_df[col].reindex(dfp[dt]).ffill()
            if close is None:
                continue

            # Write signals
            out_dir = DST_ROOT / asset; (out_dir/"figs").mkdir(parents=True, exist_ok=True)
            sig = pos.diff().fillna(0.0).replace({1.0:1,-1.0:0}).astype(int)
            df_sig = pd.DataFrame({
                "Date": dfp[dt].values,
                "signal": sig.values,
                "position": pos.astype(int).values,
                "Close": close.values,
                "strategy": strat
            })
            (out_dir / f"signals_{strat}_{F}.csv").write_text(df_sig.to_csv(index=False))

            # Metrics + fig
            close_series = pd.to_numeric(close, errors="coerce")
            close_series.index = pd.to_datetime(dfp[dt].values)
            eq, mets = to_equity(close_series, pos, F)
            plot_eq(eq, f"{asset} — {strat} — {F}", out_dir/"figs"/f"{strat}_{F}.png")
            row = {"asset": asset, "freq": F, "strategy": strat, **mets}
            if kind == "keywords":
                metrics_rows_keywords[asset].append(row)
            else:
                metrics_rows_baseline[asset].append(row)

    # If we have a summary CSV, try to augment metrics rows
    if summary_df is not None:
        sdf = summary_df.copy()
        # Try common columns
        cols_lower = {c.lower(): c for c in sdf.columns}
        # Normalize an 'asset' column if present
        if "asset" in cols_lower:
            sdf["asset_norm"] = sdf[cols_lower["asset"]].astype(str).str.upper().replace({
                "GOLD":"GOLD","BTC":"BTC","OIL":"OIL","USDCNY":"USDCNY"
            })
        else:
            sdf["asset_norm"] = None
        # Strategy column guess
        s_col = cols_lower.get("model", None) or cols_lower.get("strategy", None)
        # Return-ish column guess (if equity provided)
        ret_col = cols_lower.get("final_equity", None) or cols_lower.get("return_ann", None)
        for _, r in sdf.iterrows():
            a = r.get("asset_norm", None)
            if a not in ASSETS:
                continue
            s = str(r.get(s_col, "MODEL")) if s_col else "MODEL"
            row = {"asset": a, "freq": F, "strategy": s}
            if ret_col:
                try:
                    val = float(r[ret_col])
                    # interpret final_equity as (1+return) if > 1.1 else direct return
                    row["Return_Ann"] = (val - 1.0) if val > 1.1 else val
                except Exception:
                    pass
            # Where to place? Use path heuristic on summary location
            kind = "keywords" if is_keywords_path(summary) else "baseline"
            (metrics_rows_keywords if kind=="keywords" else metrics_rows_baseline)[a].append(row)

    # Write metrics files per asset
    for a in ASSETS:
        out_dir = DST_ROOT / a; (out_dir/"figs").mkdir(parents=True, exist_ok=True)
        mb = pd.DataFrame(metrics_rows_baseline[a]) if metrics_rows_baseline[a] else pd.DataFrame(
            columns=["asset","freq","strategy","Return_Ann","Vol_Ann","Sharpe","MaxDD"])
        mk = pd.DataFrame(metrics_rows_keywords[a]) if metrics_rows_keywords[a] else pd.DataFrame(
            columns=["asset","freq","strategy","Return_Ann","Vol_Ann","Sharpe","MaxDD"])
        # Ensure freq column is set
        if not mb.empty and "freq" not in mb.columns: mb.insert(1, "freq", F)
        if not mk.empty and "freq" not in mk.columns: mk.insert(1, "freq", F)
        (out_dir / f"metrics_baseline_{F}.csv").write_text(mb.to_csv(index=False))
        (out_dir / f"metrics_keywords_{F}.csv").write_text(mk.to_csv(index=False))

        # Optional: copy any loose figs for this asset/freq from source
        for p in pngs:
            if infer_asset(p.name) == a:
                dstp = out_dir / "figs" / p.name
                try:
                    shutil.copy2(p, dstp)
                except:
                    pass

print("✅ Wrote app-ready artefacts to:", DST_ROOT)
print("→ Copy to your repo as AppDemo/artefacts/ and redeploy.")


Mounted at /content/drive
✅ Wrote app-ready artefacts to: /content/drive/MyDrive/gt-markets/AppDemo/artefacts_from_mainnb
→ Copy to your repo as AppDemo/artefacts/ and redeploy.
